# I/O

In this example, we will show how to load various input types and to extract relevance informations

In [1]:
import numpy as np
from copy import deepcopy
from ase.io import read
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) 

from muonscripts.constants import constants
from muonscripts.io_tools.read import read_cif
from muonscripts.io_tools.read import read_from_file

import muonscripts.io_tools as ios
import muonscripts.io_tools.read_qe  as qe
import muonscripts.io_tools.read_elk  as elk

from muonscripts.io_tools.load_relax import load_relax
from muonscripts.io_tools.load_relax import save_relax_results
from muonscripts.io_tools.load_relax import load_relax_results
from muonscripts.io_tools.find_qe_files import select_groups
from muonscripts.io_tools.find_qe_files import get_relax_after_run
from muonscripts.io_tools.find_qe_files import print_relax_summary
from muonscripts.io_tools.find_qe_files import load_relax_from_folder

from muonscripts.muon_tools import contact_field
from muonscripts.muon_tools.muonic_dataframe import get_dataframe
from muonscripts.muon_tools.distortions.distortions import get_structs
from muonscripts.muon_tools.aiida_muon.clustering_report import print_clustering_summary
from muonscripts.muon_tools.aiida_muon.clustering import get_clustering_after_run_from_data

from muonscripts.muesr_tools.local_fields import pfields, rfields
from muonscripts.muesr_tools.utils import _extract_muon, get_atom_kinds

from muonscripts.muon_tools.nearby import _group_fmt
from muonscripts.muon_tools.nearby import nearby_atom
from muonscripts.muon_tools.nearby import print_muon_table
from muonscripts.muon_tools.nearby import get_group_nearby_atom



print(f"Planck Constant: {constants.H_PLANCK} J/s")
print(f"Muon Gyromagnetic Ratio: {constants.MUON_GYROMAGNETIC_RATIO} rad/(s T)")
# ROOT, sys.path

Planck Constant: 6.62607015e-34 J/s
Muon Gyromagnetic Ratio: 851615459.5446056 rad/(s T)


In [3]:
import os
dpath = './data'

load input structure formats

In [4]:
file=os.path.join(dpath, 'CaF2.cif')
read_cif(file)
read_from_file(file)

/home/misah/.virtualenvs/pyqm/lib/python3.10/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


Atoms(symbols='Ca4F8', pbc=True, cell=[5.45095, 5.45095, 5.45095], spacegroup_kinds=...)

In [5]:
file=os.path.join(dpath, 'CaF2.vasp')
read_cif(file)
read_from_file(file)

Atoms(symbols='Ca4F8', pbc=True, cell=[5.4509501457, 5.4509501457, 5.4509501457])

In [6]:
file=os.path.join(dpath, 'CaF2.vasp')
Structure.from_file(file)

Structure Summary
Lattice
    abc : 5.4509501457 5.4509501457 5.4509501457
 angles : 90.0 90.0 90.0
 volume : 161.96330486922204
      A : np.float64(5.4509501457) np.float64(0.0) np.float64(0.0)
      B : np.float64(0.0) np.float64(5.4509501457) np.float64(0.0)
      C : np.float64(0.0) np.float64(0.0) np.float64(5.4509501457)
    pbc : True True True
PeriodicSite: Ca (2.725, 2.725, 2.725) [0.5, 0.5, 0.5]
PeriodicSite: Ca (2.725, 0.0, 0.0) [0.5, 0.0, 0.0]
PeriodicSite: Ca (0.0, 2.725, 0.0) [0.0, 0.5, 0.0]
PeriodicSite: Ca (0.0, 0.0, 2.725) [0.0, 0.0, 0.5]
PeriodicSite: F (1.363, 4.088, 4.088) [0.25, 0.75, 0.75]
PeriodicSite: F (4.088, 1.363, 1.363) [0.75, 0.25, 0.25]
PeriodicSite: F (4.088, 1.363, 4.088) [0.75, 0.25, 0.75]
PeriodicSite: F (1.363, 4.088, 1.363) [0.25, 0.75, 0.25]
PeriodicSite: F (4.088, 4.088, 1.363) [0.75, 0.75, 0.25]
PeriodicSite: F (1.363, 1.363, 4.088) [0.25, 0.25, 0.75]
PeriodicSite: F (1.363, 1.363, 1.363) [0.25, 0.25, 0.25]
PeriodicSite: F (4.088, 4.088, 4.088) 

load QE input/output file

In [7]:
file = os.path.join(dpath, 'relax.in')
ios.read_qe.read_in(file)

Atoms(symbols='HCa32F64', pbc=True, cell=[10.9019002914, 10.9019002914, 10.9019002914], initial_magmoms=...)

In [8]:
file = os.path.join(dpath, 'relax.out')
ios.read_qe.read_out(file)

Atoms(symbols='HCa32F64', pbc=True, cell=[10.901900350679567, 10.901900350679567, 10.901900350679567], calculator=SinglePointDFTCalculator(...))

In [9]:
file = os.path.join(dpath, 'relax.in')
qe.read_in(file)

Atoms(symbols='HCa32F64', pbc=True, cell=[10.9019002914, 10.9019002914, 10.9019002914], initial_magmoms=...)

In [10]:
file = os.path.join(dpath, 'relax.out')
qe.read_out(file)

Atoms(symbols='HCa32F64', pbc=True, cell=[10.901900350679567, 10.901900350679567, 10.901900350679567], calculator=SinglePointDFTCalculator(...))

In [11]:
file = os.path.join(dpath, 'relax.in')
read_from_file(file, format='espresso-in')

Atoms(symbols='HCa32F64', pbc=True, cell=[10.9019002914, 10.9019002914, 10.9019002914], initial_magmoms=...)

In [12]:
file = os.path.join(dpath, 'relax.out')
atoms = read_from_file(file, format='espresso-out', index=-1)

convert ase to pymatgen

In [13]:
rlx = AseAtomsAdaptor.get_structure(atoms)
print(rlx)

Full Formula (Ca32 H1 F64)
Reduced Formula: Ca32HF64
abc   :  10.901900  10.901900  10.901900
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (97)
  #  SP            a          b          c
---  ----  ---------  ---------  ---------
  0  Ca     0.272299   0.272299   0.250002
  1  Ca     0.749781   0.749781   0.749999
  2  Ca     0.744938   0.744938   0.250004
  3  Ca     0.249817   0.249817   0.749997
  4  Ca     0.749559   0.250352   0.749998
  5  Ca     0.249065   0.750469   0.250003
  6  Ca     0.250352   0.749559   0.749998
  7  Ca     0.750469   0.249065   0.250003
  8  Ca     0.250617   0.502354   0.502636
  9  Ca     0.747644  -0.000615  -0.002641
 10  Ca     0.747649  -0.000616   0.502637
 11  Ca     0.250617   0.50236   -0.00264
 12  Ca     0.750042   0.499961  -0.000706
 13  Ca     0.253652  -0.003647   0.502124
 14  Ca     0.25365   -0.003646  -0.00211
 15  Ca     0.750043   0.499959   0.500702
 16  Ca     0.502354   0.250617   0.50263

convert pymatgen to ase

In [14]:
file=os.path.join(dpath, 'CaF2.vasp')
st = Structure.from_file(file)
atoms = AseAtomsAdaptor.get_atoms(st)
atoms

MSONAtoms(symbols='Ca4F8', pbc=True, cell=[5.4509501457, 5.4509501457, 5.4509501457])

load QE efg tensors from gipaw output file

In [15]:
file = os.path.join(dpath, 'V3Si_QE_EFG3x3x3_SITE_A.OUT')
efg_tensors = qe.read_efg(file)

efg_tensors[0]

array([[-8.62804611e+19,  0.00000000e+00, -1.14694029e+20],
       [ 0.00000000e+00, -2.97322139e+20,  0.00000000e+00],
       [-1.14694029e+20,  0.00000000e+00,  3.83602600e+20]])

load ELK output files: (geometry and EFG tensors)

In [16]:
file = os.path.join(dpath, 'MnSi_GEOMETRY.OUT')
elk.read_geom(file)

Atoms(symbols='Mn4Si4', pbc=True, cell=[4.566199937664733, 4.566199937664733, 4.566199937664733], initial_magmoms=...)

In [17]:
file = os.path.join(dpath, 'MnSi_EFG.OUT')
efg_tensors = elk.read_efg(file)

efg_tensors[0]

array([[-3.02274846e+04, -1.54534901e+20, -1.54534901e+20],
       [-1.54534901e+20, -3.12190144e+05, -1.54534901e+20],
       [-1.54534901e+20, -1.54534901e+20,  3.42417629e+05]])

remove numerical noise in efg tensors

In [18]:
remove_efg_noise=True
efg_noise_threshold=1e-8

tensors = []
for tensor in efg_tensors:
    if remove_efg_noise:
        max_element = np.max(np.abs(tensor))
        if max_element > 0:
            tensor[np.abs(tensor) < efg_noise_threshold * max_element] = 0.0
    tensors.append(tensor)

tensors[0]

array([[ 0.00000000e+00, -1.54534901e+20, -1.54534901e+20],
       [-1.54534901e+20,  0.00000000e+00, -1.54534901e+20],
       [-1.54534901e+20, -1.54534901e+20,  0.00000000e+00]])

move muon to origin, useful for hyperfine contact field calculations with QE

In [19]:
file = os.path.join(dpath, 'relax.out')
atoms = read_from_file(file, format='espresso-out', index=-1)

rlx = AseAtomsAdaptor.get_structure(atoms)

# get muon
mupos_rlx = rlx.frac_coords[rlx.atomic_numbers.index(1)]

rlx.translate_sites(
    range(rlx.num_sites), -mupos_rlx, frac_coords=True, to_unit_cell=False
)

for site in rlx:
    print(f"{site.species_string:<3} {site.a:14.9f} {site.b:14.9f} {site.c:14.9f}")

Ca     0.147298639    0.147298639   -0.000010350
Ca     0.624780375    0.624780375    0.499986135
Ca     0.619938009    0.619938009   -0.000008517
Ca     0.124817109    0.124817109    0.499984411
Ca     0.624558611    0.125352299    0.499985431
Ca     0.124064486    0.625468576   -0.000009340
Ca     0.125352299    0.624558611    0.499985431
Ca     0.625468576    0.124064486   -0.000009340
Ca     0.125617131    0.377354259    0.252623661
Ca     0.622643429   -0.125615103   -0.252653414
Ca     0.622649030   -0.125615701    0.252624146
Ca     0.125616536    0.377359862   -0.252652939
Ca     0.625041660    0.374960563   -0.250718077
Ca     0.128651806   -0.128647602    0.252111058
Ca     0.128649988   -0.128645781   -0.252122028
Ca     0.625042902    0.374959319    0.250689676
Ca     0.377354259    0.125617131    0.252623661
Ca    -0.125615103    0.622643429   -0.252653414
Ca    -0.125615701    0.622649030    0.252624146
Ca     0.377359862    0.125616536   -0.252652939
Ca    -0.128645781  

query ALL 'relax' muon site calculations from a folder

save to a file, if desired

In [20]:
folder = os.path.join(dpath, 'SZRO/query')

preview = True

pattern="*.out"
check_input = True
calc_types = "relax"


if preview:
    groups = get_relax_after_run(
        folder, 
        pattern=pattern, 
        check_input=check_input,
        calc_types=calc_types, 
        verbose=False,
    )
    print(f"\nPreview: {len(groups)} system group(s) found "
            f"(selection filters ignored in preview mode)\n")
    print_relax_summary(groups)


results = load_relax(
    path=folder,
    pattern=pattern,
    check_input=check_input,
    calc_types=calc_types,
    group_by_system=True,
    pymatgen=True,
    verbose=True,
    # **select_kwargs
)

file = 'SZRO_muonrelax_results.pkl'
file = os.path.join(dpath, file)
save_relax_results(results, file, fmt='pickle')


Preview: 1 system group(s) found (selection filters ignored in preview mode)

  H-O-Re-Sr-Zn  nat = 161  ntyp = 5  -> 12 converged runs
  ------------------------------------------
  1 distinct system(s), 12 converged run(s) total
Saved 12 result(s) to data/SZRO_muonrelax_results.pkl (pickle)


load results if present

In [21]:
file = 'SZRO_muonrelax_results.pkl'
file = os.path.join(dpath, file)

data = load_relax_results(file, fmt='pickle')

idx = 0
d = data[idx]

rlx_st = Structure.from_dict(d["rlxd_struct"])

In [22]:
print(rlx_st)

Full Formula (Sr32 Zn16 Re16 H1 O96)
Reduced Formula: Sr32Zn16Re16HO96
abc   :  11.227996  11.162597  15.774004
angles:  90.000000  90.064973  90.000000
pbc   :       True       True       True
Sites (161)
  #  SP            a          b          c
---  ----  ---------  ---------  ---------
  0  Sr     0.250704   0.260334   0.125823
  1  Sr     0.252901   0.2605     0.625221
  2  Sr     0.250981   0.76364    0.123103
  3  Sr     0.253103   0.756342   0.624224
  4  Sr     0.751958   0.260565   0.124477
  5  Sr     0.751403   0.258354   0.625733
  6  Sr     0.74971    0.761371   0.124133
  7  Sr     0.75163    0.758787   0.624806
  8  Sr    -0.001333   0.487165   0.376886
  9  Sr     0.000683   0.490662   0.87446
 10  Sr     0.002432   0.990921   0.375282
 11  Sr     0.000916   0.990218   0.875018
 12  Sr     0.512952   0.495322   0.381634
 13  Sr     0.501835   0.490607   0.87487
 14  Sr     0.500824   0.985886   0.37704
 15  Sr     0.501889   0.99051    0.875396
 16  Sr     0.243432   

cluster results

OPTIONAL: pristine structure and supercell matrix, used for the calculations
default = None

In [23]:
file_name = os.path.join(dpath, 'SZRO/SZRO_monoP.cif')
p_st = Structure.from_file(file_name)
print(p_st)

sc_matrix = np.diag([2, 2, 2])

Full Formula (Sr4 Zn2 Re2 O12)
Reduced Formula: Sr2ZnReO6
abc   :   5.614000   5.581300   7.887000
angles:  90.000000  90.065000  90.000000
pbc   :       True       True       True
Sites (20)
  #  SP         a       b       c
---  ----  ------  ------  ------
  0  Sr    0.5044  0.5239  0.2502
  1  Sr    0.0044  0.9761  0.7502
  2  Sr    0.4956  0.4761  0.7498
  3  Sr    0.9956  0.0239  0.2498
  4  Zn    0       0.5     0
  5  Zn    0.5     0       0.5
  6  Re    0.5     0       0
  7  Re    0       0.5     0.5
  8  O     0.237   0.2057  0.9725
  9  O     0.737   0.2943  0.4725
 10  O     0.763   0.7943  0.0275
 11  O     0.263   0.7057  0.5275
 12  O     0.2925  0.7306  0.9676
 13  O     0.7925  0.7694  0.4676
 14  O     0.7075  0.2694  0.0324
 15  O     0.2075  0.2306  0.5324
 16  O     0.4404  0.9906  0.2368
 17  O     0.9404  0.5094  0.7368
 18  O     0.5596  0.0094  0.7632
 19  O     0.0596  0.4906  0.2632


In [24]:
input_st = p_st.copy()
init_supc = input_st.copy()
init_supc.make_supercell(sc_matrix)

cluster = get_clustering_after_run_from_data(
    results, 
    input_st, 
    sc_matrix, 
    magmom=None
)

print('The cluster keys are:', list(cluster.keys()))

mapping = cluster["mapping"]
unique_cluster = cluster["unique_pos"]

unique_sorted = {v: i+1 for i, v in enumerate(sorted(set(mapping)))}
remapped = [unique_sorted[x] for x in mapping]

print_clustering_summary(
    cluster, 
    input_st=input_st,   # or None, if you want the supercell positions
    use_unitcell=True,   # or None, if you want the supercell positions
    # precision=5
)

The cluster keys are: ['unique_pos', 'mag_inequivalent', 'mapping']

UNIQUE MUON SITES (unitcell frac. coordinates)
  #        idx        frac_x        frac_y        frac_z       dE (eV)
----------------------------------------------------------------------
  0         r9       0.06772       0.38202       0.79190       0.00000
  1         r1       0.71618       0.80622       0.59247       0.02432
  2         r5       0.80437       0.10348       0.48491       0.03240
  3         r6       0.71269       0.20909       0.59189       0.04780
  4         r2       0.61634       0.69766       0.50053       0.07336
  5        r12       0.40882       0.10296       0.70667       0.10637
  6        r11       0.85256       0.35930       0.80577       0.11761
  7         r3       0.17541       0.04347       0.51871       0.12827
  8         r7       0.16439       0.30095       0.84954       0.14406
  9        r10       0.21274       0.30227       0.24603       0.14922
 10         r4       0.80109    

In [25]:
file = 'SZRO_clusteredmuonrelax_results.pkl'
file = os.path.join(dpath, file)
save_relax_results(unique_cluster, file, fmt='pickle')

Saved 12 result(s) to data/SZRO_clusteredmuonrelax_results.pkl (pickle)


In [26]:
file = 'SZRO_clusteredmuonrelax_results.pkl'
file = os.path.join(dpath, file)

data = load_relax_results(file, fmt='pickle')

idx = 0
d = data[idx]

rlx_st = Structure.from_dict(d["rlxd_struct"])

In [27]:
print(rlx_st)

Full Formula (Sr32 Zn16 Re16 H1 O96)
Reduced Formula: Sr32Zn16Re16HO96
abc   :  11.227996  11.162597  15.774004
angles:  90.000000  90.064973  90.000000
pbc   :       True       True       True
Sites (161)
  #  SP            a          b          c
---  ----  ---------  ---------  ---------
  0  Sr     0.257582   0.260581   0.1238
  1  Sr     0.256399   0.260193   0.625813
  2  Sr     0.252043   0.759801   0.125528
  3  Sr     0.251125   0.763148   0.624247
  4  Sr     0.748907   0.258855   0.123966
  5  Sr     0.747885   0.260649   0.627266
  6  Sr     0.751085   0.758458   0.12495
  7  Sr     0.750151   0.760451   0.625552
  8  Sr     0.003681   0.489153   0.375783
  9  Sr     0.000885   0.491736   0.87554
 10  Sr    -0.000427   0.975305   0.373378
 11  Sr     0.000221   0.988707   0.876176
 12  Sr     0.501133   0.491109   0.375235
 13  Sr     0.502321   0.491039   0.875216
 14  Sr     0.502055   0.990543   0.374812
 15  Sr     0.50179    0.990598   0.874994
 16  Sr     0.257291   0

extract muon position from relax structure

In [28]:
rlx_st_nomu, mupos_sc = _extract_muon(rlx_st, muon_pos=None, muon_label='H')
mupos_sc

array([0.03385911, 0.19101119, 0.39594981])

transform relax muon position in fractional coords of unitcell

In [29]:
mupos_uc = np.dot(mupos_sc, sc_matrix)%1  # wrap to unitcell
mupos_uc  # same as above in the dataframe

array([0.06771822, 0.38202238, 0.79189962])

from unitcell to supercell

In [30]:
np.dot(mupos_uc, np.linalg.inv(sc_matrix))%1

array([0.03385911, 0.19101119, 0.39594981])

dataframe clustered results

In [31]:
df, df_all = get_dataframe(unique_cluster, sc_matrix)
print(df_all)

   structure_id label     tot_energy       muon_position_sc  \
1            r9     A -227129.199795  [0.034, 0.191, 0.396]   
2            r1     B -227129.175477  [0.358, 0.403, 0.296]   
3            r5     C   -227129.1674  [0.402, 0.052, 0.242]   
4            r6     D -227129.151997  [0.356, 0.105, 0.296]   
5            r2     E -227129.126436   [0.308, 0.349, 0.25]   
6           r12     F -227129.093421  [0.204, 0.051, 0.353]   
7           r11     G -227129.082183   [0.426, 0.18, 0.403]   
8            r3     H -227129.071521  [0.088, 0.022, 0.259]   
9            r7     I -227129.055735   [0.082, 0.15, 0.425]   
10          r10     J -227129.050577  [0.106, 0.151, 0.123]   
11           r4     K -227129.024571  [0.401, 0.154, 0.075]   
12           r8     L -227129.000728  [0.026, 0.085, 0.497]   

         muon_position_uc delta_E  
1   [0.068, 0.382, 0.792]       0  
2   [0.716, 0.806, 0.592]      24  
3   [0.804, 0.104, 0.484]      32  
4    [0.712, 0.21, 0.592]      47  


In [32]:
print(df)
df

   structure_id label          muon_position delta_E_meV
1            r9     A  [0.068, 0.382, 0.792]           0
2            r1     B  [0.716, 0.806, 0.592]          24
3            r5     C  [0.804, 0.104, 0.484]          32
4            r6     D   [0.712, 0.21, 0.592]          47
5            r2     E    [0.616, 0.698, 0.5]          73
6           r12     F  [0.408, 0.102, 0.706]         106
7           r11     G   [0.852, 0.36, 0.806]         117
8            r3     H  [0.176, 0.044, 0.518]         128
9            r7     I     [0.164, 0.3, 0.85]         144
10          r10     J  [0.212, 0.302, 0.246]         149
11           r4     K   [0.802, 0.308, 0.15]         175
12           r8     L   [0.052, 0.17, 0.994]         199


,structure_id,label,muon_position,delta_E_meV
1,r9,A,"[0.068, 0.382, 0.792]",0
2,r1,B,"[0.716, 0.806, 0.592]",24
3,r5,C,"[0.804, 0.104, 0.484]",32
4,r6,D,"[0.712, 0.21, 0.592]",47
5,r2,E,"[0.616, 0.698, 0.5]",73
6,r12,F,"[0.408, 0.102, 0.706]",106
7,r11,G,"[0.852, 0.36, 0.806]",117
8,r3,H,"[0.176, 0.044, 0.518]",128
9,r7,I,"[0.164, 0.3, 0.85]",144
10,r10,J,"[0.212, 0.302, 0.246]",149


cluster positions based on the nearby O atoms

In [33]:
nearby_O = nearby_atom(
    'O',
    fcoords=np.asarray(
        df["muon_position"].tolist(),
        dtype=float,
    ),
    host_lattice=input_st,
    energies=df["delta_E_meV"].tolist(),
    symprec=1e-3,
)

group_nearby_O = _group_fmt('O', nearby_O)

group_nearby_O = get_group_nearby_atom(
    'O',
    fcoords=np.asarray(
        df["muon_position"].tolist(),
        dtype=float,
    ),
    host_lattice=input_st,
    energies=df["delta_E_meV"].tolist(),
    symprec=1e-3,
)

print_muon_table(group_nearby_O)


  Sites    |   Label    |    Fractional coord.     |   ΔE (meV)  
           |   A_{I}    |  (0.068, 0.382, 0.792)   |      0      
   μ-O1    |   A_{II}   |  (0.408, 0.102, 0.706)   |     106     
           |  A_{III}   |  (0.852, 0.360, 0.806)   |     117     
           |   A_{IV}   |  (0.212, 0.302, 0.246)   |     149     
---------------------------------------------------------------
           |   B_{I}    |  (0.716, 0.806, 0.592)   |      24     
   μ-O2    |   B_{II}   |  (0.616, 0.698, 0.500)   |      73     
           |  B_{III}   |  (0.176, 0.044, 0.518)   |     128     
           |   B_{IV}   |  (0.802, 0.308, 0.150)   |     175     
---------------------------------------------------------------
           |   C_{I}    |  (0.804, 0.104, 0.484)   |      32     
   μ-O3    |   C_{II}   |  (0.712, 0.210, 0.592)   |      47     
           |  C_{III}   |  (0.164, 0.300, 0.850)   |     144     
           |   C_{IV}   |  (0.052, 0.170, 0.994)   |     199     
-------------

compute spin density at muon site from QE (pp) .XSF file (up & down .xsf files)

In [34]:
up_file = 'Fe2P_QE_spinup_Astar.XSF'
up_file = 'Fe2P_QE_spinup_A.XSF'
up_file = os.path.join(dpath, up_file)
dw_file = 'Fe2P_QE_spindw_Astar.XSF'
dw_file = 'Fe2P_QE_spindw_A.XSF'
dw_file = os.path.join(dpath, dw_file)

sp_density, B_c = contact_field.field_at_muon(up_file, dw_file)

print(f'calculated spin density at muon site = {sp_density: .6e} (bohr^-3) | contact field = {B_c: .5f} (Tesla)')


calculated spin density at muon site = -8.153000e-03 (bohr^-3) | contact field = -0.42746 (Tesla)


calculated local field at the muon site: Fe2P

In [35]:
filename = 'Fe2P.cif'
filename = os.path.join(dpath, filename)
st = Structure.from_file(filename)

/home/misah/.virtualenvs/pyqm/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


In [36]:
# Identify atom kinds
# There are two distincts Fe atoms, Fe1 and Fe2 in Fe2P

atm_kinds = get_atom_kinds(st)

print("\nAtom kinds:")
for kind, indices in atm_kinds.items():
    print(f"{kind}: {indices}")


print("\nFe1 indices:")
print(atm_kinds["Fe1"])
print("\nFe2 indices:")
print(atm_kinds["Fe2"])


Atom kinds:
Fe1: [0, 1, 2]
Fe2: [3, 4, 5]
P1: [6, 7]
P2: [8]

Fe1 indices:
[0, 1, 2]

Fe2 indices:
[3, 4, 5]


In [37]:
print("\nFe1 fractional coordinates:")

for i in atm_kinds["Fe1"]:
    print(
        f"index = {i:3d}   "
        f"frac = {st.frac_coords[i]}"
    )

print("\nFe2 fractional coordinates:")

for i in atm_kinds["Fe2"]:
    print(
        f"index = {i:3d}   "
        f"frac = {st.frac_coords[i]}"
    )


Fe1 fractional coordinates:
index =   0   frac = [0.256 0.    0.   ]
index =   1   frac = [0.744 0.744 0.   ]
index =   2   frac = [0.    0.256 0.   ]

Fe2 fractional coordinates:
index =   3   frac = [0.589 0.    0.5  ]
index =   4   frac = [0.411 0.411 0.5  ]
index =   5   frac = [0.    0.589 0.5  ]


In [38]:
# Magnetic moment directions
#

Fe1_dir_001 = np.array([0.0, 0.0, 0.84])  # muB
Fe2_dir_001 = np.array([0.0, 0.0, 2.22])  # muB

print("\nMoment magnitudes:")
print("|Fe1_dir_001| =", np.linalg.norm(Fe1_dir_001))
print("|Fe2_dir_001| =", np.linalg.norm(Fe2_dir_001))


Moment magnitudes:
|Fe1_dir_001| = 0.84
|Fe2_dir_001| = 2.22


In [39]:
# Construct magnetic structures

magmoms  = np.zeros((len(st), 3))

# FM [001]
magmoms[atm_kinds["Fe1"]] = Fe1_dir_001
magmoms[atm_kinds["Fe2"]] = Fe2_dir_001

magmoms

array([[0.  , 0.  , 0.84],
       [0.  , 0.  , 0.84],
       [0.  , 0.  , 0.84],
       [0.  , 0.  , 2.22],
       [0.  , 0.  , 2.22],
       [0.  , 0.  , 2.22],
       [0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  ]])

In [40]:
mupos = np.array([0.002, 0.328, 0.501])
include_equiv = True
contact_field_value = -0.42746 # in Tesla

# calculated field, using pristine structure

fields_contribs = pfields(
    structure=st.copy(),
    magmoms=magmoms,
    muon_pos=mupos,
    cont_field=contact_field_value,  # calculated above
    include_equivalent_sites=include_equiv,
)

# field contributions container
fields_contribs

LocalFieldResults(total=array([[ 2.25621745e-03, -3.98492345e-03, -4.60537734e-01],
       [ 4.57915366e-03, -3.85201008e-05, -4.60537734e-01],
       [ 2.32293622e-03,  3.94640335e-03, -4.60537734e-01]]), dipolar=array([[ 2.25621745e-03, -3.98492345e-03, -3.79958765e-01],
       [ 4.57915366e-03, -3.85201008e-05, -3.79958765e-01],
       [ 2.32293622e-03,  3.94640335e-03, -3.79958765e-01]]), lorentz=array([[0.        , 0.        , 0.34688103],
       [0.        , 0.        , 0.34688103],
       [0.        , 0.        , 0.34688103]]), contact=array([[-0.     , -0.     , -0.42746],
       [-0.     , -0.     , -0.42746],
       [-0.     , -0.     , -0.42746]]), dipolar_tot=array([[ 2.25621745e-03, -3.98492345e-03, -3.30777336e-02],
       [ 4.57915366e-03, -3.85201008e-05, -3.30777336e-02],
       [ 2.32293622e-03,  3.94640335e-03, -3.30777336e-02]]), total_norm=array([0.4605605, 0.4605605, 0.4605605]), dipolar_norm=array([0.37998636, 0.37998636, 0.37998636]), lorentz_norm=array([0.34688

In [41]:
REF_EXP_LOCAL_FIELD = 0.3963 # Tesla

fields = ["dipolar", "lorentz", "contact", "dipolar_tot", "total"]

for i, pos in enumerate(fields_contribs.muon_positions):
    print(
        f"\nSite {i+1}  "
        f"position = ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})"
    )

    for name in fields:
        vec = getattr(fields_contribs, name)[i]
        mag = np.linalg.norm(vec)

        print(
            f"  {name:12s}: "
            f"[{vec[0]: .8f}, {vec[1]: .8f}, {vec[2]: .8f}]  (Tesla) "
            f"|B| = {mag:.8f} (Tesla)"
        )

    total_field = fields_contribs.total_norm[i]
    diff = total_field - REF_EXP_LOCAL_FIELD
    status = "Overestimation" if diff >= 0 else "Underestimation"
    accuracy = np.abs(diff) / REF_EXP_LOCAL_FIELD

    print(
        f"  Observed field     = {REF_EXP_LOCAL_FIELD:.8f} (Tesla)"
    )
    print(
        f"  Calculated field   = {total_field:.8f} (Tesla)"
    )
    print(
        f"  {status:<18} = {accuracy * 100:.1f} %"
    )


Site 1  position = (0.002, 0.328, 0.501)
  dipolar     : [ 0.00225622, -0.00398492, -0.37995876]  (Tesla) |B| = 0.37998636 (Tesla)
  lorentz     : [ 0.00000000,  0.00000000,  0.34688103]  (Tesla) |B| = 0.34688103 (Tesla)
  contact     : [-0.00000000, -0.00000000, -0.42746000]  (Tesla) |B| = 0.42746000 (Tesla)
  dipolar_tot : [ 0.00225622, -0.00398492, -0.03307773]  (Tesla) |B| = 0.03339321 (Tesla)
  total       : [ 0.00225622, -0.00398492, -0.46053773]  (Tesla) |B| = 0.46056050 (Tesla)
  Observed field     = 0.39630000 (Tesla)
  Calculated field   = 0.46056050 (Tesla)
  Overestimation     = 16.2 %

Site 2  position = (0.326, 0.998, 0.499)
  dipolar     : [ 0.00457915, -0.00003852, -0.37995876]  (Tesla) |B| = 0.37998636 (Tesla)
  lorentz     : [ 0.00000000,  0.00000000,  0.34688103]  (Tesla) |B| = 0.34688103 (Tesla)
  contact     : [-0.00000000, -0.00000000, -0.42746000]  (Tesla) |B| = 0.42746000 (Tesla)
  dipolar_tot : [ 0.00457915, -0.00003852, -0.03307773]  (Tesla) |B| = 0.03339321 

Get and include structural distortions for symmetry equivalent site/structures

In [42]:
file = 'host_mag.cif'
file = os.path.join(dpath, file)
host_ase  = read_from_file(file)

In [43]:
file = 'relax_mag.in'
file = os.path.join(dpath, file)
inp_ase  = read_from_file(file, format="espresso-in")
inp_ase2 = inp_ase.copy() 

In [44]:
file = 'relax_mag.out'
file = os.path.join(dpath, file)
rlx_ase  = read_from_file(file, format="espresso-out", index=-1)
rlx_ase2 = rlx_ase.copy()

In [45]:
host_lattice = AseAtomsAdaptor.get_structure(host_ase)
p_st = AseAtomsAdaptor.get_structure(inp_ase)
rlx_st = AseAtomsAdaptor.get_structure(rlx_ase)

In [46]:
results = get_structs(
    p_st=p_st,
    rlx_st=rlx_st,
    host_st=host_lattice,
    min_distance=0.5
)
# results

In [47]:
sc_matrix = [
    [2, 0, 0], 
    [0, 2, 0], 
    [0, 0, 3]
    ]


for i, site in enumerate(results):
    sc_frac = np.dot(site.frac_pos, sc_matrix) % 1
    sc_str = " ".join(f"{x:7.4f}" for x in site.frac_pos)
    prim_str = " ".join(f"{x:7.4f}" for x in sc_frac)
    flag = "orig" if site.is_original else ("magn " if site.is_magnetically_equivalent else "struc")
    status = "ok" if site.ok else "FAIL"
    print(f"[{i:2d}] ({sc_str}) -> ({prim_str})  {flag:5s} {status}")

[ 0] ( 0.0011  0.1641  0.1667) -> ( 0.0023  0.3282  0.5000)  orig  ok
[ 1] ( 0.1630  0.9989  0.8333) -> ( 0.3259  0.9977  0.5000)  magn  ok
[ 2] ( 0.8359  0.8370  0.1667) -> ( 0.6718  0.6741  0.5000)  magn  ok
[ 3] ( 0.0011  0.1641  0.8333) -> ( 0.0023  0.3282  0.5000)  magn  ok
[ 4] ( 0.1630  0.9989  0.1667) -> ( 0.3259  0.9977  0.5000)  magn  ok
[ 5] ( 0.8359  0.8370  0.8333) -> ( 0.6718  0.6741  0.5000)  magn  ok


In [48]:
fields = ["dipolar", "lorentz", "contact", "dipolar_tot", "total"]

total_vecs = []
for idx, site in enumerate(results):
    if site.structure is not None:
        fields_contribs2 = rfields(
            p_st=host_lattice,
            magmoms=magmoms,
            sc_mat=sc_matrix,
            r_supst=site.structure,
            cont_field=contact_field_value,
            # muon_label: str = "H",
        )

        total_vecs.append(fields_contribs2.total[0])

        sc_frac = np.dot(site.frac_pos, sc_matrix) % 1
        sc_str = " ".join(f"{x:7.4f}" for x in site.frac_pos)
        prim_str = " ".join(f"{x:7.4f}" for x in sc_frac)

        flag = "orig" if site.is_original else ("magn " if site.is_magnetically_equivalent else "struc")
        status = "ok" if site.ok else "FAIL"

        print(f"\n[{idx:2d}] ({sc_str}) -> ({prim_str})  {flag:5s} {status}")

        for i, pos in enumerate(fields_contribs2.muon_positions):

            # print(
            #     f"\nSite {i+1}  "
            #     f"position = ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})"
            # )
            

            for name in fields:
                vec = getattr(fields_contribs2, name)[i]
                mag = np.linalg.norm(vec)

                print(
                    f"  {name:12s}: "
                    f"[{vec[0]: .8f}, {vec[1]: .8f}, {vec[2]: .8f}]  (Tesla) "
                    f"|B| = {mag:.8f} (Tesla)"
                )

            total_field = fields_contribs2.total_norm[i]
            diff = total_field - REF_EXP_LOCAL_FIELD
            status = "Overestimation" if diff >= 0 else "Underestimation"
            accuracy = np.abs(diff) / REF_EXP_LOCAL_FIELD

            print(
                f"  Observed field     = {REF_EXP_LOCAL_FIELD:.8f} (Tesla)"
            )
            print(
                f"  Calculated field   = {total_field:.8f} (Tesla)"
            )
            print(
                f"  {status:<18} = {accuracy * 100:.1f} %"
            )


[ 0] ( 0.0011  0.1641  0.1667) -> ( 0.0023  0.3282  0.5000)  orig  ok
  dipolar     : [ 0.00000724, -0.00001300, -0.17994109]  (Tesla) |B| = 0.17994109 (Tesla)
  lorentz     : [ 0.00000000,  0.00000000,  0.34687449]  (Tesla) |B| = 0.34687449 (Tesla)
  contact     : [-0.00000000, -0.00000000, -0.42746000]  (Tesla) |B| = 0.42746000 (Tesla)
  dipolar_tot : [ 0.00000724, -0.00001300,  0.16693340]  (Tesla) |B| = 0.16693340 (Tesla)
  total       : [ 0.00000724, -0.00001300, -0.26052660]  (Tesla) |B| = 0.26052660 (Tesla)
  Observed field     = 0.39630000 (Tesla)
  Calculated field   = 0.26052660 (Tesla)
  Underestimation    = 34.3 %

[ 1] ( 0.1630  0.9989  0.8333) -> ( 0.3259  0.9977  0.5000)  magn  ok
  dipolar     : [ 0.00597928, -0.00562851, -0.41473004]  (Tesla) |B| = 0.41481133 (Tesla)
  lorentz     : [ 0.00000000,  0.00000000,  0.34687449]  (Tesla) |B| = 0.34687449 (Tesla)
  contact     : [-0.00000000, -0.00000000, -0.42746000]  (Tesla) |B| = 0.42746000 (Tesla)
  dipolar_tot : [ 0.0059

In [49]:
total_vecs = np.array(total_vecs)

# 1. Component-wise Mean & Standard Deviation
vec_mean = np.mean(total_vecs, axis=0)
vec_std = np.std(total_vecs, axis=0, ddof=1)  # Sample standard deviation (N - 1)

# 2. Individual Vector Magnitudes
magnitudes = np.linalg.norm(total_vecs, axis=1)
mag_mean = np.mean(magnitudes)
mag_std = np.std(magnitudes, ddof=1)

# 3. Reference Field Diagnostics
mag_of_mean = np.linalg.norm(vec_mean)

# Difference and accuracy for vector mean magnitude
diff_mean = mag_of_mean - REF_EXP_LOCAL_FIELD
status_mean = "Overestimation" if diff_mean >= 0 else "Underestimation"
accuracy_mean = (np.abs(diff_mean) / REF_EXP_LOCAL_FIELD) * 100

# Difference and accuracy for scalar mean of magnitudes
diff_scalar = mag_mean - REF_EXP_LOCAL_FIELD
status_scalar = "Overestimation" if diff_scalar >= 0 else "Underestimation"
accuracy_scalar = (np.abs(diff_scalar) / REF_EXP_LOCAL_FIELD) * 100

# --- Clean Formatted Print Output ---
divider = "=" *70

print(f"\n{divider}")
print(f"{'VECTOR FIELD STATISTICS (Tesla)':^62}")
print(f"{divider}\n")

# Individual Vectors Table
print("Individual Field Vectors and Magnitudes:")
print("-" * 70)
print(f"{'Index':<7} | {'Bx (T)':^13} | {'By (T)':^13} | {'Bz (T)':^13} | {'|B| (T)':^11}")
print("-" * 70)
for idx, (vec, mag) in enumerate(zip(total_vecs, magnitudes)):
    print(f" #{idx:<5d} | {vec[0]:13.6e} | {vec[1]:13.6e} | {vec[2]:13.6e} | {mag:11.6f}")
print("-" * 70)

# Component-wise Summary
print("\n--- Component-wise Averages ---")
print(f"Mean Vector [Bx, By, Bz] : [{vec_mean[0]:13.6e}, {vec_mean[1]:13.6e}, {vec_mean[2]:13.6e}] T")
print(f"Std Dev     [Bx, By, Bz] : [{vec_std[0]:13.6e}, {vec_std[1]:13.6e}, {vec_std[2]:13.6e}] T")

# Magnitude Summary
print("\n--- Scalar Magnitude Summary ---")
print(f"Magnitude of Mean Vector (||<B>||) : {mag_of_mean:.8f} T")
print(f"Mean of Magnitudes       (<||B||>) : {mag_mean:.8f} T")
print(f"Std Dev of Magnitudes              : {mag_std:.8f} T")

# Experimental Comparison
print("\n--- Comparison to Experimental Reference ---")
print(f"  Observed field     = {REF_EXP_LOCAL_FIELD:.8f} (Tesla)")
print(f"  Calculated field   = {mag_of_mean:.8f} (Tesla)")
print(f"  {status_mean:<18} = {accuracy_mean:.1f} %")
print(f"{divider}\n")


               VECTOR FIELD STATISTICS (Tesla)                

Individual Field Vectors and Magnitudes:
----------------------------------------------------------------------
Index   |    Bx (T)     |    By (T)     |    Bz (T)     |   |B| (T)  
----------------------------------------------------------------------
 #0     |  7.235161e-06 | -1.299798e-05 | -2.605266e-01 |    0.260527
 #1     |  5.979277e-03 | -5.628513e-03 | -4.953155e-01 |    0.495384
 #2     |  8.976261e-06 |  1.491810e-05 | -4.977352e-01 |    0.497735
 #3     |  2.706365e-04 | -8.998199e-04 | -4.418278e-01 |    0.441829
 #4     | -1.745155e-05 |  7.435918e-08 | -4.973329e-01 |    0.497333
 #5     |  1.500321e-03 | -8.427606e-03 | -4.952827e-01 |    0.495357
----------------------------------------------------------------------

--- Component-wise Averages ---
Mean Vector [Bx, By, Bz] : [ 1.291499e-03, -2.492324e-03, -4.480035e-01] T
Std Dev     [Bx, By, Bz] : [ 2.369328e-03,  3.639909e-03,  9.440987e-02] T

--- Sca